This notebook is intended to encompass the entire workflow of the Bridge Builder 1.0 Project

Author: Jason Sanchez

# Figure 2

## Creating stability.csv to compare CTU Mutations

In [ ]:
import pandas as pd
import numpy as np
import constants as cs
import re
import os
import glob
from tqdm import tqdm
import structure as st
import distances as ds

In [ ]:
pdbs = glob.glob("./skempi/PDBs/*.pdb")

In [ ]:
skempi_df = pd.read_csv("./skempi/skempi_v2.csv", delimiter=";")

In [ ]:
def deltaG(BA, temp):
    return (8.314/4184)*(temp) * np.log(np.array(BA))

In [ ]:
def broken_SB(entry):
    WT = cs.AA_1_3[entry[0]]
    MUT = cs.AA_1_3[entry[-1]]

    if WT in cs.CHARGED and MUT in cs.NEUTRAL:
        return 1
    
    else:
        return 0

In [ ]:
def stabalizing(entry1, entry2):
    if entry1 < entry2:
        return 1
    else:
        return 0

In [ ]:
skempi_df

In [ ]:
skempi_df = skempi_df[skempi_df["Mutation(s)_cleaned"].str.contains(",") == False]
skempi_df.dropna(subset=["Affinity_wt_parsed", "Affinity_mut_parsed", "Temperature"]).reset_index(drop=True)
skempi_df = skempi_df[skempi_df.Temperature != "298(assumed)"].reset_index(drop=True)
skempi_df.Temperature = skempi_df.Temperature.astype("float")

In [ ]:
skempi_df["deltaG_wt"] = skempi_df.apply(lambda row: deltaG(row["Affinity_wt_parsed"], row["Temperature"]), axis=1)
skempi_df["deltaG_mut"] = skempi_df.apply(lambda row: deltaG(row["Affinity_mut_parsed"], row["Temperature"]), axis=1)

In [ ]:
stability_df = skempi_df[["#Pdb", "Mutation(s)_cleaned", "iMutation_Location(s)", "deltaG_wt", "deltaG_mut"]].reset_index(drop=True)
stability_df = stability_df.groupby(by=["#Pdb", "Mutation(s)_cleaned", "iMutation_Location(s)"], as_index=False).aggregate({"deltaG_wt": "mean", "deltaG_mut" : "mean"})

In [ ]:
stability_df["potential_broken_sb"] = stability_df["Mutation(s)_cleaned"].apply(broken_SB) 

In [ ]:
potential_sb_df = stability_df[stability_df.potential_broken_sb == 1]

In [ ]:
potential_sb_df["stabalizing"] = potential_sb_df.apply(lambda x: stabalizing(x.deltaG_mut, x.deltaG_wt), axis=1)
final_df = potential_sb_df.reset_index(drop=True)
final_df = final_df.drop(["potential_broken_sb"], axis=1)

In [ ]:
final_df["broke_sb"] = 0
final_df["sb_partner"] = np.nan

In [ ]:
for i in tqdm(range(len(final_df))):
    cur_row = final_df.iloc[i, :]
    
    pdb = cur_row["#Pdb"][0:4]
    
    mut_info = cur_row["Mutation(s)_cleaned"]

    c = mut_info[1]
    wt = cs.AA_1_3[mut_info[0]]
    mut = cs.AA_1_3[mut_info[-1]]
    ri = int(re.findall(r'\d+', mut_info)[0])
    
    input_file = os.path.join("./skempi/PDBs/", pdb + ".pdb")

    try:
        cmplx = st.COMPLEX(input_file)
        info, distances = ds.all_salt_bridges(cmplx)
        
        for i2 in range(len(info)):
            cur_row = info[i2, :]

            if (cur_row[0] != cur_row[3] and cur_row[0] == c and int(cur_row[2]) == ri):
                final_df.at[i, "broke_sb"] = 2
                final_df.at[i, "sb_partner"] = cur_row[3] + "_" + cur_row[5]
                break
            elif (cur_row[0] != cur_row[3] and cur_row[3] == c and int(cur_row[5]) == ri):
                final_df.at[i, "broke_sb"] = 2
                final_df.at[i, "sb_partner"] = cur_row[0] + "_" + cur_row[2]
                break
            elif (cur_row[0] == c and int(cur_row[2]) == ri):
                final_df.at[i, "broke_sb"] = 1
                final_df.at[i, "sb_partner"] = cur_row[3] + "_" + cur_row[5]
                break
            elif (cur_row[3] == c and int(cur_row[5]) == ri):
                final_df.at[i, "broke_sb"] = 1
                final_df.at[i, "sb_partner"] = cur_row[0] + "_" + cur_row[2]
                break


    except Exception as e:
        print(e)

In [ ]:
final_df = final_df.dropna(subset=["deltaG_wt", "deltaG_mut"])
final_df.reset_index(drop=True)
final_df = final_df.drop("sb_partner", axis=1)
final_df.to_csv("stability.csv", index=False)
final_df

## Creating Figure 2 from stability.csv

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
stability_df = pd.read_csv("stability.csv")

In [ ]:
interface_locs = ["COR", "SUP", "RIM"]
non_interface_locs = ["INT", "SUR"]

In [ ]:
interprotein_sb_df = stability_df[stability_df.broke_sb == 2]
intraprotein_interface_sb_df = stability_df[(stability_df.broke_sb == 1) & (stability_df["iMutation_Location(s)"]).isin(interface_locs)]
intraprotein_non_interface_sb_df = stability_df[(stability_df.broke_sb == 1) & (stability_df["iMutation_Location(s)"]).isin(non_interface_locs)]
no_sb_interface_df = stability_df[(stability_df.broke_sb == 0) & (stability_df["iMutation_Location(s)"]).isin(interface_locs)]
no_sb_non_interface_df = stability_df[(stability_df.broke_sb == 0) & (stability_df["iMutation_Location(s)"]).isin(non_interface_locs)]



all_ddg = np.array(stability_df.deltaG_mut - stability_df.deltaG_wt)
interprotein_ddg = np.array(interprotein_sb_df.deltaG_mut - interprotein_sb_df.deltaG_wt)
intraprotein_interface_ddg = np.array(intraprotein_interface_sb_df.deltaG_mut - intraprotein_interface_sb_df.deltaG_wt)
intraprotein_non_interface_ddg = np.array(intraprotein_non_interface_sb_df.deltaG_mut - intraprotein_non_interface_sb_df.deltaG_wt)
no_interface_ddg = np.array(no_sb_interface_df.deltaG_mut - no_sb_interface_df.deltaG_wt)
no_non_interface_ddg = np.array(no_sb_non_interface_df.deltaG_mut - no_sb_non_interface_df.deltaG_wt)


all_ddg = all_ddg[~np.isnan(all_ddg)]
interprotein_ddg = interprotein_ddg[~np.isnan(interprotein_ddg)]
intraprotein_interface_ddg = intraprotein_interface_ddg[~np.isnan(intraprotein_interface_ddg)]
intraprotein_non_interface_ddg = intraprotein_non_interface_ddg[~np.isnan(intraprotein_non_interface_ddg)]
no_interface_ddg = no_interface_ddg[~np.isnan(no_interface_ddg)]
no_non_interface_ddg = no_non_interface_ddg[~np.isnan(no_non_interface_ddg)]

all_dg = stability_df.deltaG_wt
all_dg = all_dg[~np.isnan(all_dg)]


### All Boxplots

In [ ]:
plt.figure().set_figwidth(10)
plt.boxplot([all_ddg, interprotein_ddg, intraprotein_interface_ddg, intraprotein_non_interface_ddg, no_interface_ddg, no_non_interface_ddg], showfliers=False, widths=0.4, labels=["All CTU", "Interprotein SB", "Intraprotein SB (INTF)", "Intraprotein SB (NINTF)", "No SB (INTF)", "No SB (NINTF)"])
plt.ylabel("\u0394\u0394G (Kcal/mol)")
plt.tight_layout()
plt.show()

In [ ]:
len(all_ddg), len(interprotein_ddg), len(intraprotein_interface_ddg), len(intraprotein_non_interface_ddg), len(no_interface_ddg), len(no_non_interface_ddg)

### Percentiles

In [ ]:
percentiles = [25, 50, 75]
fig_1_dict = {"all_ddg": all_ddg, "interprotein_ddg": interprotein_ddg, 
              "intraprotein_interface_ddg": intraprotein_interface_ddg,
             "intraprotein_non_interface_ddg": intraprotein_non_interface_ddg,
             "no_interface_ddg": no_interface_ddg, 
              "no_non_interface_ddg": no_non_interface_ddg}

In [ ]:
for partition, data in fig_1_dict.items():
    for percentile in percentiles:
        value = np.percentile(data, percentile)
        print("The {}th percentile for {} is: {}".format(percentile, partition, value))

### Individual Boxplots

In [ ]:
plt.figure().set_figwidth(3)
plt.boxplot([all_dg], showfliers=False, widths=0.4, labels=["All Mutations"])
plt.ylabel("\u0394G (Kcal/mol)")
plt.tight_layout()
plt.savefig("./figures/all_mutations_deltaG.png", dpi = 350)
plt.show()

In [ ]:
plt.figure().set_figwidth(3)
plt.boxplot([all_ddg], showfliers=False, widths=0.4, labels=["All Mutations"])
plt.ylabel("\u0394\u0394G (Kcal/mol)")
plt.tight_layout()
plt.ylim(-3,6)
plt.savefig("./figures/all_mutations_deltadeltaG.png", dpi = 350)
plt.show()

In [ ]:
plt.figure().set_figwidth(5)
plt.boxplot([interprotein_ddg, intraprotein_interface_ddg, no_interface_ddg], showfliers=False, widths=0.4, labels=["Interprotein SBB", "Intraprotein SBB", "NSBB"])
plt.ylabel("\u0394\u0394G (Kcal/mol)")
plt.tight_layout()
plt.ylim(-3,6)
plt.savefig("./figures/interface_mutations_deltadeltaG.png", dpi = 350)
plt.show()

In [ ]:
plt.figure().set_figwidth(3)
plt.boxplot([intraprotein_non_interface_ddg, no_non_interface_ddg], showfliers=False, widths=0.4, labels=["SBB", "NSB"])
plt.ylabel("\u0394\u0394G (Kcal/mol)")
plt.tight_layout()
plt.ylim(-1,2)
plt.savefig("./figures/noninterface_mutations_deltadeltaG.png", dpi = 350)
plt.show()

# Figures 3, 4, and 5

## We need to find all SBs in SKEMPI

In [1]:
import subprocess
import glob
import os
import tqdm

All of the salt bridges in the SKEMPI database are saved in vmd_sb where the folder name is the name of the PDB entry.

In [ ]:
pdbfiles = glob.glob("skempi/PDBs/*.pdb")
cur_dir = os.getcwd()
pdbfiles = [os.path.join(cur_dir, x) for x in pdbfiles]
for f in tqdm(pdbfiles):
    PDB_id = os.path.split(f)[1].split(".")[0]
    output_path = os.path.join("./vmd_sb", PDB_id)
    if not os.path.isdir(output_path):
        os.mkdir(output_path)
    
    subprocess.run(["vmd", f, "-e", "templete.tcl"], capture_output=True)
    
    sb_files = glob.glob("*.dat")
    
    for sf in sb_files:
        os.rename(sf, os.path.join(output_path, sf))

## Generating the Train and Test Sets

The files, train.csv and test.csv, are compilations of the salt bridge information in vmd_sb. <br>
Half of the proteins are in train.csv (118) and the other half (118) are in test.csv.

In [2]:
import os, glob
import constants as cs
import re
import pandas as pd
from sklearn.model_selection import train_test_split

In [3]:
RANDOM_STATE = 12
vmd_sb_folder = os.path.abspath("./vmd_sb/")
vmd_subfolders = os.listdir(vmd_sb_folder)

In [4]:
d = []
for pdbid in vmd_subfolders:
    cur_path = os.path.join(vmd_sb_folder, pdbid)
    cur_files = glob.glob(cur_path + "/*.dat")
    for file in cur_files:
        with open(file, "r") as f:
            data = f.readlines()
        
        dist = float(data[0].split()[1])
                
        filename = os.path.split(file)[1].split(".")[0]
        
        components = filename.split("-")
                
        chain1 = components[1][-1]
        chain2 = components[2][-1]
        
        resname1 = components[1][0:3]
        resname2 = components[2][0:3]

        resid1 = int(re.findall(r'\d+', components[1])[0])
        resid2 = int(re.findall(r'\d+', components[2])[0])
        
        result = [pdbid, chain1, resname1, resid1, chain2, resname2, resid2, dist]
        d.append(result)

In [5]:
colnames = ["pdb", "chain1", "resname1", "resid1", "chain2", "resname2", "resid2", "sb_dist"]
vmd_df = pd.DataFrame(data=d, columns=colnames)

In [6]:
for i in range(len(vmd_df)):
    cur_row = vmd_df.iloc[i, :]
    c1 = cur_row["chain1"]
    n1 = cur_row["resname1"]
    i1 = cur_row["resid1"] 
    
    c2 = cur_row["chain2"]
    n2 = cur_row["resname2"]
    i2 = cur_row["resid2"]
    
    if n1 in cs.POSITIVE:
        continue
    else:
        vmd_df.at[i, "chain1"] = c2
        vmd_df.at[i, "resname1"] = n2
        vmd_df.at[i, "resid1"] = i2
        
        vmd_df.at[i, "chain2"] = c1
        vmd_df.at[i, "resname2"] = n1
        vmd_df.at[i, "resid2"] = i1

In [7]:
vmd_df = vmd_df[vmd_df.chain1 != vmd_df.chain2].reset_index(drop=True)
vmd_df.to_csv("vmd_sb.csv", index=False)
vmd_df = pd.read_csv("./vmd_sb.csv")

In [8]:
pdbs = sorted(vmd_df.pdb.unique().tolist())
train, test = train_test_split(pdbs, train_size=0.5, random_state=RANDOM_STATE)

In [9]:
def assign_train_test(input):
    if input in train:
        return 0
    else:
        return 1

In [10]:
vmd_df["split"] = vmd_df.pdb.apply(assign_train_test)

In [11]:
vmd_df[vmd_df.split == 0].to_csv("train.csv", index=False)
vmd_df[vmd_df.split == 1].to_csv("test.csv", index=False)

In [12]:
len(train), len(test)

(118, 118)

## Mutating Residues at Random

Run the following script to generate mutations.sh from train.csv and test.csv. The shell script file will be parallelized by the TACC LAUNCHER utility. The slurm file is mutate.slurm

In [ ]:
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")
dfs = [train_df, test_df]

train_set_mutations_dir = "train_set_mutations"
test_set_mutations_dir = "test_set_mutations"
dirs = [train_set_mutations_dir, test_set_mutations_dir]

mut_residues = sorted(list(cs.NEUTRAL))
skemi_dir = "./skempi/PDBs"

out_filenames = ["train_set.sh", "test_set.sh"]

for df, dir, out in zip(dfs, dirs, out_filenames):
    if not os.path.isdir(dir):
        os.mkdir(dir)

    if os.path.isfile(out):
        os.remove(out)

    for i in range(len(df)):
        choose_mut = i%2
        cur_row = df.iloc[i, :]
        pdb_id = cur_row["pdb"]

        if choose_mut == 0:
            c1 = cur_row["chain1"]
            n1 = cur_row["resname1"]
            i1 = cur_row["resid1"] 
            
            c2 = cur_row["chain2"]
            n2 = cur_row["resname2"]
            i2 = cur_row["resid2"]

        else:
            c2 = cur_row["chain1"]
            n2 = cur_row["resname1"]
            i2 = cur_row["resid1"] 
            
            c1 = cur_row["chain2"]
            n1 = cur_row["resname2"]
            i1 = cur_row["resid2"]


        pdb_in_path = os.path.join(skemi_dir, pdb_id + ".pdb")

        for mut_residue in mut_residues:
            pdb_out_path = os.path.join(dir, "{}_{}_{}_{}_{}_{}.pdb".format(pdb_id, c1, i1, c2, i2, mut_residue))

            line = "PyMOLMutateAminoAcids.py -m {}:{}{}{} -i {} -o {}\n".format(c1, n1, i1,mut_residue, pdb_in_path, pdb_out_path)

            with open(out, "a") as f:
                f.write(line)

data = data2 = ""
with open(out_filenames[0]) as fp:
    data = fp.read()
 
with open(out_filenames[1]) as fp:
    data2 = fp.read()
 
data += "\n"
data += data2
 
with open ('mutations.sh', 'w') as fp:
    fp.write(data)


## Using the Train Set

The following script, gen_train_distances_sh.py, generates a shell script train_distances.sh which can be parallelized by the slurm script train_distances.slurm.

In [ ]:
import os, glob

train_set_mutations_dir = "train_set_mutations"
train_distances_sh = "train_distances.sh"
train_set_distances_dir = "train_set_distances"

if os.path.isfile(train_distances_sh):
    os.remove(train_distances_sh)

filenames = glob.glob(train_set_mutations_dir + "/*.pdb")

with open(train_distances_sh, "a") as f:
    for file in filenames:
        f.write("python get_distances.py {} {}\n".format(file, train_set_distances_dir))

The train_distances.sh script calls the following python script, get_distances.py, to find the residue distances of two residues.

In [ ]:
import os
import structure as st
import distances as ds
import sys
import pickle

pdbfile = sys.argv[1]
output_folder = sys.argv[2]
file_info = os.path.split(pdbfile)[1].split(".")[0].split("_")
d = []

pdb = file_info[0]
c1 = file_info[1]
i1 = int(file_info[2])
c2 = file_info[3]
i2 = int(file_info[4])
n = file_info[5]

cmplx = st.COMPLEX(pdbfile)
res1 = cmplx[c1][i1]
res2 = cmplx[c2][i2]

original_file = os.path.join("skempi/PDBs/", pdb + ".pdb")
original_cmplx = st.COMPLEX(original_file)
or_res1 = original_cmplx[c1][i1]
or_res2 = original_cmplx[c2][i2]

d.append(ds.minimum_dist(res1, res2))
d.append(ds.centroid_dist(res1, res2))
d.append(ds.sidechain_centroid_dist(res1, res2))
d.append(ds.sidechain_dist(res1, res2))
d.append(ds.alpha_dist(res1, res2))
d.append(ds.NO_dist(or_res1, or_res2))

with open("{}/{}_{}_{}_{}_{}_{}.pickle".format(output_folder, pdb, c1, i1, c2, i2, n), "wb") as f:
        pickle.dump(d, f)

## Collecting Train Set Data

Implemented as collect_train_distances.py on TACC. The data need to be collected into train_distances.csv.

In [ ]:
import pickle, glob, os
import pandas as pd
from tqdm import tqdm

distance_files = glob.glob("train_set_distances/*.pickle")
distance_list = []
colnames = ["pdb", "chain1", "resid1", "chain2", "resid2", "name", "md", "cd", "sccd", "scd", "ad", "nod"]

for file in tqdm(distance_files):
    with open(file, "rb") as f:
        data = pickle.load(f)
    file_info = os.path.split(file)[1].split(".")[0].split("_")
    pdb = file_info[0]
    c1 = file_info[1]
    i1 = int(file_info[2])
    c2 = file_info[3]
    i2 = int(file_info[4])
    n = file_info[5]
    
    cur_info = [pdb, c1, i1, c2, i2, n]

    distance_list.append(cur_info + data)

df = pd.DataFrame(columns=colnames, data = distance_list)

df.to_csv("train_distances.csv", index=False)

## Analyzing the Train Set

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
train_distance_df = pd.read_csv("train_distances.csv")

percentiles = [25, 50, 75, 90]
distances = ["nod", "cd", "sccd", "ad","md", "scd"]

for percentile in percentiles:
    print("The Percentile is: {}".format(percentile))
    for distance in distances:
        value = round(np.percentile(train_distance_df[distance], percentile), 3)
        print("The distance is {} and the value is {}".format(distance, value)) 

## Creating Figure 3

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

train_distance_df = pd.read_csv("train_distances.csv")

fig, ax =  plt.subplots(2,3, figsize =(12, 6))

top_xlim = (0, 17)
bottom_xlim = (0,17)

top_ylim = (0, 500)
bottom_ylim = (0, 500)

bins = 41


ax[0,0].hist(train_distance_df[train_distance_df.nod <= 3.2].nod, bins=bins)
# ax[0,0].axvline(nod, color = "black")
ax[0,1].hist(train_distance_df.cd, bins=bins)
# ax[0,1].axvline(cd_percentile, color = "black")
ax[0,2].hist(train_distance_df.sccd, bins=bins)
# ax[0,2].axvline(sccd_percentile, color = "black")

ax[1,0].hist(train_distance_df.ad, bins=bins)
# ax[1,0].axvline(acd_percentile, color = "black")
ax[1,1].hist(train_distance_df.md, bins=bins)
# ax[1,1].axvline(md_percentile, color = "black")
ax[1,2].hist(train_distance_df.scd, bins=bins)
# ax[1,2].axvline(scd_percentile, color = "black")

ax[0,0].title.set_text("Nitrogen-Oxygen Distance")
ax[1,1].title.set_text("Minimum Distance")
ax[1,2].title.set_text("Side Chain Distance")
ax[1,0].title.set_text("Alpha Carbon Distance")
ax[0,1].title.set_text("Centroid Distance")
ax[0,2].title.set_text("Side Chain Centroid Distance")

ax[0,0].set_ylabel("Count")
ax[1,0].set_ylabel("Count")

ax[1,0].set_xlabel("Distance (Å)")
ax[1,1].set_xlabel("Distance (Å)")
ax[1,2].set_xlabel("Distance (Å)")



for i in range(2):
    for j in range(3):
        if i ==0:
            ax[i,j].set_xlim(top_xlim)
            ax[i,j].set_xticks(range(0, 17, 2))
            ax[i,j].set_ylim(top_ylim)
        else:
            ax[i,j].set_xlim(bottom_xlim)
            ax[i,j].set_xticks(range(0, 17, 2))
            ax[i,j].set_ylim(bottom_ylim)
            
        

ax[0,0].set_xlim(0,5)
ax[0,0].set_xticks(range(0,5, 1))
plt.tight_layout()
plt.savefig("./figures/all_train_distances.png", dpi=350)

## Using the Test Set

The following script, gen_test_distances_sh.py, generates a shell script test_distances.sh which can be parallelized by the slurm script test_distances.slurm.

In [ ]:
import os, glob

test_set_mutations_dir = "test_set_mutations"
test_distances_sh = "test_distances.sh"
test_set_distances_dir = "test_set_distances"
test_set_errors_dir = "test_distance_errors"

if os.path.isfile(test_distances_sh):
    os.remove(test_distances_sh)

filenames = glob.glob(test_set_mutations_dir + "/*.pdb")

with open(test_distances_sh, "a") as f:
    for file in filenames:
        error_file = os.path.split(file)[1].split(".")[0]
        f.write("python get_within_distances.py {} {} 2> {}/{}.err\n".format(file, test_set_distances_dir, test_set_errors_dir, error_file))

The test_distances.sh script calls the following python script, get_within_distances.py, to find the residue distances of two residues.

In [ ]:
import os
import structure as st
import distances as ds
import sys
import pickle
import time

pdbfile = sys.argv[1]
output_folder = sys.argv[2]
file_info = os.path.split(pdbfile)[1].split(".")[0].split("_")

pdb = file_info[0]
c1 = file_info[1]
i1 = int(file_info[2])
c2 = file_info[3]
i2 = int(file_info[4])
n = file_info[5]

cmplx = st.COMPLEX(pdbfile)
cutoffs = {"minimum_distance": 6.254, "sidechain_distance": 6.351,
                "alpha_carbon_distance": 12.095, "centroid_distance": 10.128,
                "sidechain_centroid_distance": 8.553}

output_dict = dict()
for name, func in ds.res_func_dict.items():
    cutoff = cutoffs[name]
    start = time.time()
    info, distances = ds.within_distance(cmplx, c1, i1, func=func, cutoff=cutoff, uncharged=False)
    end = time.time()
    output_dict[name] = (info, distances, end - start)

with open("{}/{}_{}_{}_{}_{}_{}.pickle".format(output_folder, pdb, c1, i1, c2, i2, n), "wb") as f:
        pickle.dump(output_dict, f)


## Collecting Test Set Data

Implemented as collect_test_distances.py on TACC. The output is compiled into the file test_distances.csv.

In [ ]:
import pandas as pd
import os, glob
import distances as ds
import pickle
from tqdm import tqdm

test_distances = glob.glob("./test_set_distances/*.pickle")

colnames = ["pdb", "chain1_mut", "resid1_mut", "chain2_charged", "resid2_charged", "mutation", "res_distance_name", "found_chain", "found_name", "found_id", "distance", "time"]
data =  []

for filename in tqdm(test_distances):
    components = os.path.split(filename)[1].split(".")[0].split("_")
    pdb = components[0]
    c1 = components[1]
    i1 = components[2]
    c2 = components[3]
    i2 = components[4]
    n = components[5]

    if c1 == c2:
        continue

    with open(filename, "rb") as f:
        result_dict = pickle.load(f)

    for key in ds.res_func_dict.keys():
        result = result_dict[key]
        info = result[0]
        distances = result[1]
        time = result[2]

        for i in range(len(info)):
            cur_datum = [pdb, c1, i1, c2, i2, n, key, info[i, 0], info[i, 1], info[i, 2], distances[i], time]
            data.append(cur_datum)

final_df = pd.DataFrame(columns=colnames, data=data)
final_df.to_csv("test_distances.csv", index=False)


Calculations need to be perfomed on the test distance data to more easily obtain test statistics. The output is test_statistics.csv.

In [ ]:
import pandas as pd
from tqdm import tqdm

data = []
colnames = ["pdb", "c1m", "i1m", "c2c", "i2c", "mutation", "distance", "present", "position", "total_found", "time"]

test_distances_df = pd.read_csv("./test_distances.csv")

test_distances_df = test_distances_df[test_distances_df.chain1_mut != test_distances_df.found_chain].reset_index(drop=True)

distances = test_distances_df.res_distance_name.unique().tolist()
mutations = test_distances_df[["pdb", "chain1_mut", "resid1_mut", "chain2_charged", "resid2_charged", "mutation"]].drop_duplicates().values
result_dict = dict()

for distance in distances:
    for cur_mutation in tqdm(mutations):
        pdb = cur_mutation[0]
        chain1mut = cur_mutation[1]
        resid1mut = int(cur_mutation[2])
        chain2charged = cur_mutation[3]
        resid2charged = int(cur_mutation[4])
        mutation = cur_mutation[5]

        cur_df = test_distances_df[(test_distances_df.pdb == pdb) & (test_distances_df.chain1_mut == chain1mut) & (test_distances_df.resid1_mut == resid1mut) & (test_distances_df.chain2_charged == chain2charged) & \
                        (test_distances_df.resid2_charged == resid2charged) & (test_distances_df.mutation == mutation) & (test_distances_df.res_distance_name == distance)]
        
        cur_df = cur_df.sort_values(by="distance", ascending=True).reset_index(drop=True)

        if len(cur_df[(cur_df.found_chain == chain2charged) & (cur_df.found_id == resid2charged)]) >= 1:
            pos = cur_df[(cur_df.found_chain == chain2charged) & (cur_df.found_id == resid2charged)].index.values[0]
            n = len(cur_df)
            time = np.mean(np.array(cur_df.time))

            data.append([pdb, chain1mut, resid1mut, chain2charged, resid2charged, mutation, distance, 1, pos, n, time])

        else:

            data.append([pdb, chain1mut, resid1mut, chain2charged, resid2charged, mutation, distance, 0, None, None, None])

test_statistics_df = pd.DataFrame(columns=colnames, data=data)
test_statistics_df.to_csv("test_statistics.csv", index=False)

## Analyzing the Test Set

## Individual Statistics

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

test_statistics_df = pd.read_csv("./test_statistics.csv")

md_suc = 100 * sum(test_statistics_df[test_statistics_df.distance == "minimum_distance"].present == 1) / len(test_statistics_df[test_statistics_df.distance == "minimum_distance"].present == 1)
cd_suc = 100 * sum(test_statistics_df[test_statistics_df.distance == "centroid_distance"].present == 1) / len(test_statistics_df[test_statistics_df.distance == "centroid_distance"].present == 1)
sccd_suc = 100 *sum(test_statistics_df[test_statistics_df.distance == "sidechain_centroid_distance"].present == 1) / len(test_statistics_df[test_statistics_df.distance == "sidechain_centroid_distance"].present == 1)
acd_suc = 100 * sum(test_statistics_df[test_statistics_df.distance == "alpha_carbon_distance"].present == 1) / len(test_statistics_df[test_statistics_df.distance == "alpha_carbon_distance"].present == 1)
scd_suc = 100 * sum(test_statistics_df[test_statistics_df.distance == "sidechain_distance"].present == 1) / len(test_statistics_df[test_statistics_df.distance == "sidechain_distance"].present == 1)

 
  
# creating the dataset
data = {'MD':md_suc, 'CD':cd_suc, 'SCCD':sccd_suc,
        'ACD':acd_suc, 'SCD':scd_suc}
distances1 = list(data.keys()) 
percentages1 = list(data.values())
  
fig = plt.figure(figsize = (7, 5))
 
# creating the bar plot
plt.bar(distances1, percentages1, color =['#f79256', '#fbd1a2', '#7dcfb6', '#00b2ca', '#1d4e89'],
        width = 0.4)

plt.ylim(70,95)
 
plt.xlabel("Residue Distance")
plt.ylabel("Percent of Salt Bridges Identified")
plt.tight_layout()
plt.savefig("./figures/found.png", dpi = 350)
plt.show()

In [ ]:
md_pos = sum(test_statistics_df[test_statistics_df.distance == "minimum_distance"].position == 0) / len(test_statistics_df[test_statistics_df.distance == "minimum_distance"].position == 0)
cd_pos = sum(test_statistics_df[test_statistics_df.distance == "centroid_distance"].position == 0 ) / len(test_statistics_df[test_statistics_df.distance == "centroid_distance"].position == 0)
sccd_pos = sum(test_statistics_df[test_statistics_df.distance == "sidechain_centroid_distance"].position == 0) / len(test_statistics_df[test_statistics_df.distance == "sidechain_centroid_distance"].position == 0)
acd_pos = sum(test_statistics_df[test_statistics_df.distance == "alpha_carbon_distance"].position == 0) / len(test_statistics_df[test_statistics_df.distance == "alpha_carbon_distance"].position == 0)
scd_pos = sum(test_statistics_df[test_statistics_df.distance == "sidechain_distance"].position == 0) / len(test_statistics_df[test_statistics_df.distance == "sidechain_distance"].position == 0)

md_pos_suc = md_pos * md_suc
cd_pos_suc = cd_pos * cd_suc
sccd_pos_suc = sccd_pos * sccd_suc
acd_pos_suc = acd_pos * acd_suc
scd_pos_suc = scd_pos * scd_suc


import matplotlib.pyplot as plt
 
  
# creating the dataset
data = {'MD':md_pos_suc, 'CD':cd_pos_suc, 'SCCD':sccd_pos_suc,
        'ACD':acd_pos_suc, 'SCD':scd_pos_suc}
distances = list(data.keys()) 
percentages = list(data.values())
  
fig = plt.figure(figsize = (7, 5))
 
# creating the bar plot
plt.bar(distances, percentages, color =['#f79256', '#fbd1a2', '#7dcfb6', '#00b2ca', '#1d4e89'],
        width = 0.4)

plt.ylim(50,95)
 
plt.xlabel("Residue Distance")
plt.ylabel("Percent of Salt Bridges Ranked 1")
plt.tight_layout()
plt.savefig("./figures/pos.png", dpi = 350)
plt.show()

## Creating Figure 4

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

MD_vals = [md_suc,md_pos*100]
CD_vals = [cd_suc, cd_pos * 100]
SCCD_vals = [sccd_suc, sccd_pos * 100]
ACD_vals = [acd_suc, acd_pos*100]
SCD_vals = [scd_suc, scd_pos*100]


pos_label = ["WTSBI", "WTSBF"]

 
# set width of bar
barWidth = 0.15
fig = plt.subplots(figsize =(12, 8))
 
 
# Set position of bar on X axis
br1 = np.arange(len(MD_vals))
br2 = [x + barWidth for x in br1]
br3 = [x + barWidth for x in br2]
br4 = [x + barWidth for x in br3]
br5 = [x + barWidth for x in br4]
 
# Make the plot
plt.bar(br1, CD_vals, color ='#f79256', width = barWidth,
        edgecolor ='grey', label ='CD')#f79256
plt.bar(br2, SCCD_vals, color ='#fbd1a2', width = barWidth,
        edgecolor ='grey', label ='SCCD')#fbd1a2
plt.bar(br3, ACD_vals, color ='#7dcfb6', width = barWidth,
        edgecolor ='grey', label ='ACD')#7dcfb6
plt.bar(br4, MD_vals, color ='#00b2ca', width = barWidth,
        edgecolor ='grey', label ='MD')#f79256
plt.bar(br5, SCD_vals, color ='#1d4e89', width = barWidth,
        edgecolor ='grey', label ='SCD')
 
# Adding Xticks
# plt.xlabel('Metric', fontsize = 15)
plt.ylabel('Percentage', fontsize = 15)
plt.ylim(40, 100)
plt.xticks([r + 2*barWidth for r in range(len(MD_vals))], pos_label, fontsize = 15)
plt.yticks(fontsize = 15)
 
plt.legend(fontsize = "15")
plt.tight_layout()
plt.savefig("./figures/suc_pos.png", dpi = 350)
plt.show()

## Creating Figure 5

In [ ]:

MD_time = test_statistics_df[test_statistics_df.distance == "minimum_distance"].time
CD_time = test_statistics_df[test_statistics_df.distance == "centroid_distance"].time
SCCD_time = test_statistics_df[test_statistics_df.distance == "sidechain_centroid_distance"].time
ACD_time = test_statistics_df[test_statistics_df.distance == "alpha_carbon_distance"].time
SCD_time = test_statistics_df[test_statistics_df.distance == "sidechain_distance"].time

MD_time = MD_time[~np.isnan(MD_time)]
CD_time = CD_time[~np.isnan(CD_time)]
SCCD_time = SCCD_time[~np.isnan(SCCD_time)]
ACD_time = ACD_time[~np.isnan(ACD_time)]
SCD_time = SCD_time[~np.isnan(SCD_time)]




plt.figure().set_figwidth(8)
plt.boxplot([CD_time, SCCD_time, ACD_time, MD_time, SCD_time], showfliers=False, widths=0.25, labels=["CD", "SCCD", "ACD", "MD", "SCD"])
plt.ylabel("Time (s)")
plt.yscale("log")
plt.tight_layout()
plt.savefig("./figures/time.png", dpi = 350)
plt.show()

In [ ]:
print(len(CD_time), len(SCCD_time), len(ACD_time), len(MD_time), len(SCD_time))

In [ ]:
import numpy as np

percentiles = [25, 50, 75]
times = {"cd": CD_time, "sccd":SCCD_time, "acd":ACD_time, "md":MD_time, "scd": SCD_time}

for name, data in times.items():
    for percentile in percentiles:
        value = np.percentile(data, percentile)
        print("The {} percentile value for {} distance is {}".format(percentile, name, value))